# Decay chain (Bateman equations)

*nuclear · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/decay_chain)

A parent decays into a radioactive daughter, which decays into a stable end product. The three rate equations form one linear system $\dot N_i = A_{ij} N_j$, solved exactly by the matrix exponential $e^{At}$ for every time at once and checked against RK4. The default chain is the medical generator $^{99}$Mo → $^{99m}$Tc → $^{99}$Tc.

**Vector form**

$$\dot{\mathbf N} = \mathsf A\,\mathbf N \;\;\Rightarrow\;\; \mathbf N(t) = e^{\mathsf A t}\,\mathbf N(0),\qquad \mathsf A = \begin{pmatrix}-\lambda_1 & 0 & 0\\ \lambda_1 & -\lambda_2 & 0\\ 0 & \lambda_2 & 0\end{pmatrix}$$

**Index (tensor) form**

$$\dot N_i = A_{ij}\,N_j,\qquad N_{ti} = \big[e^{A\,t}\big]_{tij}\,N_j(0),\qquad a_i = \lambda_i N_i$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\lambda_i$ | `decay_constants(half_lives)` | (3,) | $\ln 2/T_i$, with $\lambda_3 = 0$ for the stable end |
| $A_{ij}$ | `bateman_matrix(lam)` | (3, 3) | $-\lambda_i$ on the diagonal, $+\lambda_i$ just below it |
| $A_{ij} N_j$ | `einsum('ij,j->i', a, n)` | (3,) | repeated $j$ ⇒ summed: the right-hand side used by RK4 |
| $[e^{A t}]_{tij}$ | `torch.linalg.matrix_exp(t[:, None, None] * a)` | (T, 3, 3) | one matrix exponential per time, batched over $t$ |
| $N_{ti}$ | `einsum('tij,j->ti', U, n0)` | (T, 3) | populations at every time from the initial vector |
| $a_i$ | `n * lam` | (T, 3) | activity: decays per hour of each member |

Each column of $\mathsf A$ sums to zero, so $\sum_i N_i$ never changes: every decay moves one nucleus down the chain. For two members the exact (Bateman) solution is $N_2 = N_1(0)\,\frac{\lambda_1}{\lambda_2 - \lambda_1}\,(e^{-\lambda_1 t} - e^{-\lambda_2 t})$. When the daughter is shorter-lived ($\lambda_2 > \lambda_1$) the activities lock together: the ratio $a_2/a_1$ tends to $\lambda_2/(\lambda_2-\lambda_1)$ (transient equilibrium), and to 1 when the parent lives much longer (secular equilibrium, try a parent half-life of 1000 h). Make the daughter longer-lived and no equilibrium forms. Simplifications: $^{99}$Mo really feeds $^{99m}$Tc in only about 88 % of decays, and $^{99}$Tc (half-life 211 000 years) is treated as stable.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/decay_chain/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import activity, bateman_matrix


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    lam = tensors.decay_constants([params["half_life_parent"], params["half_life_daughter"]], dev)  # (3,)
    a = bateman_matrix(lam)  # (3, 3)
    n0 = tensors.initial_populations(1000.0, len(lam), dev)
    t = tensors.time_axis(params["t_max"], 300, dev)

    n = solve.exact(a, n0, t)  # (T, 3)
    t_rk4, n_rk4 = solve.rk4(a, n0, params["t_max"], params["rk4_steps"])
    act = activity(n, lam)  # (T, 3)

    l1, l2 = lam[0].item(), lam[1].item()
    print(f"λ1 = {l1:.4f} 1/h   λ2 = {l2:.4f} 1/h")
    print(f"max |N_exact − N_RK4| at t_max: {(n_rk4[-1] - solve.exact(a, n0, t_rk4[-1:])[0]).abs().max().item():.2e}")
    print(f"total N at t_max: {n[-1].sum().item():.6f} (started with {n0.sum().item():.0f})")
    print(f"activity ratio a2/a1 at t_max: {act[-1, 1].item() / act[-1, 0].item():.4f}")
    if l2 > l1:
        print(f"equilibrium prediction λ2/(λ2 − λ1) = {l2 / (l2 - l1):.4f}")
    else:
        print("daughter lives longer than parent: no equilibrium, the ratio keeps growing")
    names = ["parent", "daughter", "stable end"]
    return plot.animate(t, n, t_rk4, n_rk4, act, names, out_dir / "decay_chain.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/decay_chain/equation.py
r"""Physics → tensor translation.

A decay chain 1 → 2 → … → M (the last member stable) obeys the Bateman equations

    dN_1/dt = -λ_1 N_1,      dN_i/dt = λ_{i-1} N_{i-1} - λ_i N_i,

which is one linear system with a constant matrix A:

    \dot N_i = A_{ij} N_j        (j summed),
    A_{ii} = -λ_i,   A_{i+1,i} = +λ_i,   every other entry 0.

Its exact solution is the matrix exponential N_i(t) = [e^{A t}]_{ij} N_j(0). The activity of member i
(decays per unit time) is a_i = λ_i N_i.
"""

import torch


def bateman_matrix(lam: torch.Tensor) -> torch.Tensor:
    r"""A_{ij} from the decay constants λ_i (last one 0 for a stable end).   (M,) → (M, M)"""
    loss = torch.diag(-lam)  # -λ_i on the diagonal: member i decays away
    gain = torch.diag(lam[:-1], diagonal=-1)  # +λ_i one row below: …into member i + 1
    return loss + gain


def rhs(n: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
    r"""\dot N_i = A_{ij} N_j.   n: (M,) → (M,)"""
    return torch.einsum("ij,j->i", a, n)


def propagator(a: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    r"""[e^{A t}]_{ij} for every time at once.   a: (M, M), t: (T,) → (T, M, M)"""
    return torch.linalg.matrix_exp(t[:, None, None] * a)


def activity(n: torch.Tensor, lam: torch.Tensor) -> torch.Tensor:
    r"""a_i = λ_i N_i (no sum).   n: (..., M) → (..., M)"""
    return n * lam


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/decay_chain/tensors.py
"""Build the concrete tensors: decay constants, initial populations and the time axis."""

import math

import torch


def decay_constants(half_lives: list[float], device: torch.device) -> torch.Tensor:
    """λ_i = ln 2 / T_i for the unstable members, followed by λ = 0 for the stable end: shape (M,)."""
    lam = [math.log(2.0) / t for t in half_lives] + [0.0]
    return torch.tensor(lam, dtype=torch.float64, device=device)


def initial_populations(n0: float, n_members: int, device: torch.device) -> torch.Tensor:
    """N_i(0): a pure sample of the parent, nothing else yet: shape (M,)."""
    n = torch.zeros(n_members, dtype=torch.float64, device=device)
    n[0] = n0
    return n


def time_axis(t_max: float, n_times: int, device: torch.device) -> torch.Tensor:
    """t_k evenly spaced from 0 to t_max: shape (T,)."""
    return torch.linspace(0.0, t_max, n_times, dtype=torch.float64, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/decay_chain/solve.py
"""Solve the Bateman system twice: exactly with the matrix exponential, and with RK4 as a check."""

import torch

from problems._common.integrators import integrate

from .equation import propagator, rhs


def exact(a: torch.Tensor, n0: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """N_i(t) = [e^{A t}]_{ij} N_j(0) for all times in one batched matrix_exp: shape (T, M)."""
    return torch.einsum("tij,j->ti", propagator(a, t), n0)


def rk4(a: torch.Tensor, n0: torch.Tensor, t_max: float, n_steps: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Step dN/dt = A N with classical Runge–Kutta: (times (S+1,), N (S+1, M))."""
    dt = t_max / n_steps
    traj = integrate(lambda n, _t: rhs(n, a), n0, dt, n_steps)
    times = torch.arange(n_steps + 1, dtype=n0.dtype, device=n0.device) * dt
    return times, traj


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/decay_chain/plot.py
"""Animate the populations N_i(t) (exact curves, RK4 dots) and the activities approaching equilibrium."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    t: torch.Tensor,
    n: torch.Tensor,
    t_rk4: torch.Tensor,
    n_rk4: torch.Tensor,
    act: torch.Tensor,
    names: list[str],
    path: Path,
) -> Path:
    t, n, act = t.cpu().numpy(), n.cpu().numpy(), act.cpu().numpy()  # (T,), (T, M), (T, M)
    t_rk4, n_rk4 = t_rk4.cpu().numpy(), n_rk4.cpu().numpy()
    n_members = n.shape[1]

    fig, (ax_n, ax_a) = plt.subplots(1, 2, figsize=(8.5, 4))
    lines, dots = [], []
    for i in range(n_members):
        lines.append(ax_n.plot([], [], color=ACCENT[i], lw=2, label=names[i])[0])
        dots.append(ax_n.plot([], [], "o", color=ACCENT[i], ms=3.5, mfc="white")[0])
    ax_n.set_xlim(0, t[-1])
    ax_n.set_ylim(0, n.max() * 1.05)
    ax_n.set_xlabel("t (h)")
    ax_n.set_ylabel("number of nuclei $N_i$")
    ax_n.set_title("lines: $e^{At}$   circles: RK4", fontsize=9)
    ax_n.legend(loc="center right", fontsize=8)

    act_lines = [
        ax_a.plot([], [], color=ACCENT[i], lw=2, label=f"{names[i]} activity")[0] for i in range(n_members - 1)
    ]
    ax_a.set_yscale("log")
    ax_a.set_xlim(0, t[-1])
    ax_a.set_ylim(max(act[:, 0].min() * 0.2, act.max() * 1e-4), act.max() * 2)
    ax_a.set_xlabel("t (h)")
    ax_a.set_ylabel(r"activity $\lambda_i N_i$ (1/h)")
    ax_a.legend(loc="lower left", fontsize=8)
    ratio_text = ax_a.text(0.97, 0.95, "", transform=ax_a.transAxes, ha="right", va="top", fontsize=9)

    every = max(1, len(t_rk4) // 25)  # show every few RK4 steps as circles

    def update(step: int):
        for i in range(n_members):
            lines[i].set_data(t[: step + 1], n[: step + 1, i])
            shown = t_rk4 <= t[step]
            dots[i].set_data(t_rk4[shown][::every], n_rk4[shown][::every, i])
        for i, line in enumerate(act_lines):
            line.set_data(t[: step + 1], act[: step + 1, i])
        if act[step, 0] > 0:
            ratio_text.set_text(f"t = {t[step]:.0f} h\n$a_2/a_1$ = {act[step, 1] / act[step, 0]:.3f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 72), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'half_life_parent': 66.0,
    'half_life_daughter': 6.0,
    't_max': 150.0,
    'rk4_steps': 200
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.decay_chain")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.decay_chain.main")
problem_dir = Path("problems/nuclear/decay_chain")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
